# Titanic Dataset
This noebook goes through the modelling of the famous Titanic Dataset available on [Kaggle](https://www.kaggle.com/competitions/titanic). This is the second notebook for this 'lab' assignment.

Task have been provided my ChatGPT, the answers are my own.

## Task 04 - Baseline Model

1. Create train/test split (`train_test_split` with `random_state=42`, `test_size=0.2`).

2. Build a baseline pipeline using `ColumnTransformer` + `Pipeline`:

    - Numeric transformer: `imputer` (median) + `StandardScaler`

    - Categorical transformer: `OneHotEncoder(handle_unknown='ignore')`

3. Use a simple model: `LogisticRegression (solver='saga' or 'liblinear', set max_iter=1000)`.

4. Fit pipeline on training set, predict on test set.

5. Compute metrics: Accuracy, ROC-AUC, Precision, Recall, F1. Display confusion matrix.

Deliverable: `Pipeline` object, test metrics printed.

Checkpoint goal: baseline F1 ~ around 0.6–0.75 (this is a sanity check; numbers vary).

In [1]:
import pandas as pd

df = pd.read_csv('../data/train.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
import src.data_utils as du
X_inputs, y_inputs = du.preprocess(df)
X_inputs, y_inputs

(     Pclass   Age     Fare Embarked  Gender  Title  FamilySize  IsAlone
 0         3  22.0   7.2500        S       1     Mr           2        1
 1         1  38.0  71.2833        C       0    Mrs           2        1
 2         3  26.0   7.9250        S       0   Miss           1        0
 3         1  35.0  53.1000        S       0    Mrs           2        1
 4         3  35.0   8.0500        S       1     Mr           1        0
 ..      ...   ...      ...      ...     ...    ...         ...      ...
 886       2  27.0  13.0000        S       1  Other           1        0
 887       1  19.0  30.0000        S       0   Miss           1        0
 888       3  28.0  23.4500        S       0   Miss           4        1
 889       1  26.0  30.0000        C       1     Mr           1        0
 890       3  32.0   7.7500        Q       1     Mr           1        0
 
 [891 rows x 8 columns],
 0      0
 1      1
 2      1
 3      1
 4      0
       ..
 886    0
 887    1
 888    0
 889   

In [4]:
X_inputs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pclass      891 non-null    int64  
 1   Age         891 non-null    float64
 2   Fare        891 non-null    float64
 3   Embarked    891 non-null    object 
 4   Gender      891 non-null    int64  
 5   Title       891 non-null    object 
 6   FamilySize  891 non-null    int64  
 7   IsAlone     891 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_inputs, y_inputs, random_state=42, test_size=0.2)

In [6]:
X_train, y_train

(     Pclass   Age      Fare Embarked  Gender Title  FamilySize  IsAlone
 331       1  45.5   28.5000        S       1    Mr           1        0
 733       2  23.0   13.0000        S       1    Mr           1        0
 382       3  32.0    7.9250        S       1    Mr           1        0
 704       3  26.0    7.8542        S       1    Mr           2        1
 813       3   6.0   31.2750        S       0  Miss           7        1
 ..      ...   ...       ...      ...     ...   ...         ...      ...
 106       3  21.0    7.6500        S       0  Miss           1        0
 270       1  28.0   31.0000        S       1    Mr           1        0
 860       3  41.0   14.1083        S       1    Mr           3        1
 435       1  14.0  120.0000        S       0  Miss           4        1
 102       1  21.0   77.2875        S       1    Mr           2        1
 
 [712 rows x 8 columns],
 331    0
 733    0
 382    0
 704    0
 813    0
       ..
 106    1
 270    0
 860    0
 435   

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [8]:
num_cols = ['Age',	'Fare',	'Gender', 'FamilySize', 'IsAlone']
cat_cols = ['Pclass',	'Embarked', 'Title']

In [9]:
numeric_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    [
        ('numeric', numeric_preprocessor, num_cols),
        ("categorical", categorical_preprocessor, cat_cols),
    ]
)

In [10]:
from sklearn.linear_model import LogisticRegression

model_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver="liblinear", max_iter=1000))
])    

In [11]:
model_lr.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
preds = model_lr.predict(X_train)
preds

array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0,

In [13]:
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

def display_scores(predictions, targets):
    print(f"Accuracy Score: {accuracy_score(targets, predictions):.4f}")
    print("")
    print(f"ROC-AUC Score: {roc_auc_score(targets, predictions):.4f}")
    print("")
    print(f"Precision Score: {precision_score(targets, predictions):.4f}")
    print("")
    print(f"Recall Score: {recall_score(targets, predictions):.4f}")
    print("")
    print(f"F1 Score: {f1_score(targets, predictions):.4f}")
    print("")
    print(f"Confusion Matrix: {confusion_matrix(targets, predictions)}")


In [14]:
print("Training metrics")
display_scores(preds, y_train)

preds = model_lr.predict(X_test)
print("Testing metrics")
display_scores(preds, y_test)

Training metrics
Accuracy Score: 0.8371

ROC-AUC Score: 0.8183

Precision Score: 0.8089

Recall Score: 0.7425

F1 Score: 0.7743

Confusion Matrix: [[397  47]
 [ 69 199]]
Testing metrics
Accuracy Score: 0.8101

ROC-AUC Score: 0.8022

Precision Score: 0.7778

Recall Score: 0.7568

F1 Score: 0.7671

Confusion Matrix: [[89 16]
 [18 56]]


We did achieve our baseline goal of getting an F1 score of ~ 0.6-0.75 by getting 0.76 on our test set.

## Task 05 - Try One Stronger Model

1. Try `RandomForestClassifier` with a small grid of hyperparameters (n_estimators 50/100, max_depth 5/10). Use `GridSearchCV` with 3-fold CV (or `RandomizedSearchCV` with 10 iterations).

2. Use the same pipeline structure but swap the estimator.

3. Report the best CV score and test set metrics (same metrics as baseline).

4. Save the best model with `joblib.dump(best_pipeline, 'models/best_model.joblib')`.

Deliverable: best model saved and test metrics in `02_modeling.ipynb`.

Hint: Keep grids small to save time. Use `n_jobs=-1` if your machine allows.

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [16]:
model_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])  

In [17]:
params_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5,10]
}

In [18]:
grid = GridSearchCV(model_rf, params_grid, n_jobs=-1, scoring='f1', verbose=1)

In [19]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [5, 10], 'classifier__n_estimators': [50, 100]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,None
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric', ...), ('categorical', ...)]"


In [20]:
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'classifier__max_depth': 5, 'classifier__n_estimators': 50}
Best CV score: 0.7495491419823699


In [21]:
grid.cv_results_

{'mean_fit_time': array([0.52419558, 0.92610593, 0.35714102, 0.73508234]),
 'std_fit_time': array([0.13231301, 0.22148232, 0.07156298, 0.12162033]),
 'mean_score_time': array([0.08603024, 0.07359333, 0.05797801, 0.07129469]),
 'std_score_time': array([0.06621075, 0.01057001, 0.0391856 , 0.04183338]),
 'param_classifier__max_depth': masked_array(data=[5, 5, 10, 10],
              mask=[False, False, False, False],
        fill_value=999999),
 'param_classifier__n_estimators': masked_array(data=[50, 100, 50, 100],
              mask=[False, False, False, False],
        fill_value=999999),
 'params': [{'classifier__max_depth': 5, 'classifier__n_estimators': 50},
  {'classifier__max_depth': 5, 'classifier__n_estimators': 100},
  {'classifier__max_depth': 10, 'classifier__n_estimators': 50},
  {'classifier__max_depth': 10, 'classifier__n_estimators': 100}],
 'split0_test_score': array([0.75268817, 0.73913043, 0.72727273, 0.74747475]),
 'split1_test_score': array([0.7961165 , 0.7961165 , 0.

In [22]:
best_model = grid.best_estimator_

In [ ]:
preds = best_model.predict(X_train)
print('Training metrics')
display_scores(preds, y_train)

preds = best_model.predict(X_test)
print('Testing metrics')
display_scores(preds, y_test)

Training metrics
Accuracy Score: 0.8581

ROC-AUC Score: 0.8345

Precision Score: 0.8646

Recall Score: 0.7388

F1 Score: 0.7968

Confusion Matrix: [[413  31]
 [ 70 198]]
Test metrics
Accuracy Score: 0.8212

ROC-AUC Score: 0.8097

Precision Score: 0.8088

Recall Score: 0.7432

F1 Score: 0.7746

Confusion Matrix: [[92 13]
 [19 55]]


In [24]:
import joblib

joblib.dump(best_model, '../models/best_random_forest.joblib')

['../models/best_random_forest.joblib']

## Task 06 - Explainability & Feature Importance

1. For Logistic Regression, show coefficients (map back to feature names) and list top positive/negative features.

2. For Random Forest, show feature importances (use permutation importance or built-in).

3. Add 2–3 sentences interpreting what features are most predictive (e.g., "Being female increases survival probability").

Deliverable: short section in notebook: feature importance + interpretation.

In [28]:
feature_names = model_lr.named_steps['preprocessor'].get_feature_names_out()
feature_names

array(['numeric__Age', 'numeric__Fare', 'numeric__Gender',
       'numeric__FamilySize', 'numeric__IsAlone', 'categorical__Pclass_1',
       'categorical__Pclass_2', 'categorical__Pclass_3',
       'categorical__Embarked_C', 'categorical__Embarked_Q',
       'categorical__Embarked_S', 'categorical__Title_Master',
       'categorical__Title_Miss', 'categorical__Title_Mr',
       'categorical__Title_Mrs', 'categorical__Title_Other'], dtype=object)

In [29]:
weights = model_lr.named_steps['classifier'].coef_[0]
weights

array([-0.24370919,  0.24947969, -0.94523581, -0.80612384,  0.14333801,
        0.77683292,  0.23296056, -0.80398473,  0.29556148,  0.09488941,
       -0.18464213,  1.73485266, -0.3796546 , -1.1401723 ,  0.63238174,
       -0.64159876])

In [40]:
coeff_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": weights
})

coeff_df.sort_values(by="coefficient", ascending=False)

,feature,coefficient
11,categorical__Title_Master,1.734853
5,categorical__Pclass_1,0.776833
14,categorical__Title_Mrs,0.632382
8,categorical__Embarked_C,0.295561
1,numeric__Fare,0.249480
6,categorical__Pclass_2,0.232961
4,numeric__IsAlone,0.143338
9,categorical__Embarked_Q,0.094889
10,categorical__Embarked_S,-0.184642
0,numeric__Age,-0.243709


According to the weights of the logistic regression model, looks like being `Master`, which I assume refers to young boys, has the most positive correlation to survival. I find this very interesting, as this isn't something on the top of my mind!

In [36]:
weights_rf = best_model.named_steps['classifier'].feature_importances_
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
feature_names

array(['numeric__Age', 'numeric__Fare', 'numeric__Gender',
       'numeric__FamilySize', 'numeric__IsAlone', 'categorical__Pclass_1',
       'categorical__Pclass_2', 'categorical__Pclass_3',
       'categorical__Embarked_C', 'categorical__Embarked_Q',
       'categorical__Embarked_S', 'categorical__Title_Master',
       'categorical__Title_Miss', 'categorical__Title_Mr',
       'categorical__Title_Mrs', 'categorical__Title_Other'], dtype=object)

In [41]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": weights_rf
})

importance_df.sort_values(by="coefficient", ascending=False)

,feature,coefficient
2,numeric__Gender,0.247958
13,categorical__Title_Mr,0.198750
1,numeric__Fare,0.108013
3,numeric__FamilySize,0.069266
0,numeric__Age,0.068153
7,categorical__Pclass_3,0.068139
14,categorical__Title_Mrs,0.067077
12,categorical__Title_Miss,0.046836
5,categorical__Pclass_1,0.043125
6,categorical__Pclass_2,0.021461


Now, this is much more easier to explain as compare to what we found to be the case for logistic regression. For Random Forest Classifier, `Gender` is very important. Most likely to explain what we found in our EDA, that men had disproportionately low survival rates. It is even more evident by the fact that the Title `Master` has the second highest importance coefficient. This reaffirms the `Gender` bias in the survival rate.